# Assessment Scoring and Evidence-Based Feedback
Scores a learner response across six dimensions and generates actionable feedback using **LangChain** and **LangGraph** with **Llama 3.3 (HuggingFace)**.

In [5]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# Load environment variables
load_dotenv()
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN") or os.getenv("HF_TOKEN")

# Initialize HuggingFace Llama-3.3-70B conversational LLM
endpoint = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.3-70B-Instruct",
    task="conversational",
    huggingfacehub_api_token=hf_token
)
llm = ChatHuggingFace(llm=endpoint)
print("LLM Loaded:", endpoint.repo_id)


LLM Loaded: meta-llama/Llama-3.3-70B-Instruct


In [6]:
from typing import TypedDict
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, END

# 1. Define State
class AssessmentState(TypedDict):
    question: str
    response: str
    score: str
    feedback: str

# 2. Scoring Node
def score_node(state: AssessmentState):
    prompt = PromptTemplate.from_template(
        "Score the response from 0-100 on accuracy, completeness, reasoning, communication, evidence, and professionalism. "
        "Return JSON with score and dimension_scores.\nQuestion: {q}\nResponse: {r}"
    )
    res = llm.invoke(prompt.format(q=state["question"], r=state["response"]))
    return {"score": res.content}

# 3. Feedback Node
def feedback_node(state: AssessmentState):
    prompt = PromptTemplate.from_template(
        "Give concise evidence-based feedback and one concrete next action based on this score.\nScore: {score}"
    )
    res = llm.invoke(prompt.format(score=state["score"]))
    return {"feedback": res.content}

# 4. Build and Compile LangGraph Pipeline
graph = StateGraph(AssessmentState)
graph.add_node("score", score_node)
graph.add_node("feedback", feedback_node)
graph.set_entry_point("score")
graph.add_edge("score", "feedback")
graph.add_edge("feedback", END)
app = graph.compile()

# 5. Execute Pipeline
sample_q = "Explain the difference between supervised and unsupervised learning."
sample_r = "Supervised learning uses labeled datasets to train models for prediction, while unsupervised learning discovers hidden patterns in unlabeled data."

output = app.invoke({"question": sample_q, "response": sample_r})
print("=== SCORE OUTPUT ===\n", output["score"])
print("\n=== FEEDBACK OUTPUT ===\n", output["feedback"])


=== SCORE OUTPUT ===
 Here's the score:

```json
{
  "score": 90,
  "dimension_scores": {
    "accuracy": 95,
    "completeness": 80,
    "reasoning": 90,
    "communication": 95,
    "evidence": 70,
    "professionalism": 95
  }
}
```

Here's a breakdown of the scoring:

* **Accuracy (95)**: The response accurately distinguishes between supervised and unsupervised learning.
* **Completeness (80)**: The response provides a brief overview of the difference, but lacks specific examples, details, or nuances of each type of learning.
* **Reasoning (90)**: The response demonstrates a clear understanding of the fundamental concepts, but doesn't delve into the underlying reasoning or implications of each approach.
* **Communication (95)**: The response is concise, clear, and easy to understand, making it accessible to a broad audience.
* **Evidence (70)**: The response doesn't provide any concrete evidence, examples, or references to support the claims, which reduces its credibility.
* **Prof